# 02 - Latency Analysis

Deep dive into latency distributions, comparisons, and reference baselines.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.rcParams.update({"font.family": "serif", "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})

try:
    %store -r df
    print(f"Loaded {len(df):,} events")
except:
    np.random.seed(42)
    n = 10000
    df = pd.DataFrame({
        "algorithm": np.random.choice(["RSA-2048", "ECDSA-P256", "Kyber-768"], n),
        "operation": np.random.choice(["encrypt", "decrypt", "sign", "verify"], n),
        "latency_us": np.random.lognormal(6, 0.5, n).astype(int),
    })


In [ ]:
# Overall latency distribution
lat = df["latency_us"].values

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# PDF
ax1 = axes[0, 0]
ax1.hist(lat, bins=100, density=True, alpha=0.7, edgecolor="white")
ax1.axvline(np.mean(lat), color="red", linestyle="--", label=f"Mean: {np.mean(lat):.0f}")
ax1.axvline(np.median(lat), color="green", linestyle="--", label=f"Median: {np.median(lat):.0f}")
ax1.set_xlabel("Latency (μs)")
ax1.set_ylabel("Density")
ax1.set_title("Latency Distribution (PDF)")
ax1.legend()

# CDF
ax2 = axes[0, 1]
sorted_lat = np.sort(lat)
cdf = np.arange(1, len(sorted_lat) + 1) / len(sorted_lat)
ax2.plot(sorted_lat, cdf, linewidth=2)
for p, color in [(50, "green"), (90, "orange"), (99, "red")]:
    val = np.percentile(lat, p)
    ax2.axvline(val, linestyle="--", color=color, label=f"p{p}: {val:.0f}")
ax2.set_xlabel("Latency (μs)")
ax2.set_ylabel("Cumulative Probability")
ax2.set_title("Latency CDF")
ax2.legend()

# Log scale
ax3 = axes[1, 0]
ax3.hist(lat, bins=100, alpha=0.7, edgecolor="white")
ax3.set_yscale("log")
ax3.set_xlabel("Latency (μs)")
ax3.set_ylabel("Frequency (log)")
ax3.set_title("Latency (Log Scale)")

# Tail
ax4 = axes[1, 1]
sf = np.maximum(1 - cdf, 1e-10)
ax4.semilogy(sorted_lat, sf, linewidth=2)
ax4.set_xlabel("Latency (μs)")
ax4.set_ylabel("P(Latency > x)")
ax4.set_title("Tail Distribution")

plt.tight_layout()
plt.show()


In [ ]:
# Latency by algorithm
if "algorithm" in df.columns:
    print("Latency by Algorithm:")
    summary = df.groupby("algorithm")["latency_us"].agg(["count", "mean", "std"])
    summary["p50"] = df.groupby("algorithm")["latency_us"].quantile(0.50)
    summary["p99"] = df.groupby("algorithm")["latency_us"].quantile(0.99)
    print(summary.round(1))


# 02 - Latency Analysis

Deep analysis of cryptographic operation latency distributions.

## Objectives
- Compute latency distributions
- Compare against reference baselines
- Identify tail latency patterns
- Analyze per-operation latency


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

EXPERIMENT_ID = "exp_2025_0101_001"  # Change this
DATA_PATH = f"../data/{EXPERIMENT_ID}/merged/merged.parquet"

df = pd.read_parquet(DATA_PATH)
print(f"Loaded {len(df):,} records")


In [ ]:
# Compute percentiles
percentiles = [50, 75, 90, 95, 99, 99.9]
latency = df["latency_us"]

print("Latency Percentiles (μs):")
for p in percentiles:
    print(f"  p{p:5.1f}: {np.percentile(latency, p):10.2f}")
print(f"\n  Mean:  {latency.mean():10.2f}")
print(f"  Std:   {latency.std():10.2f}")


In [ ]:
# Distribution plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram
axes[0, 0].hist(latency, bins=100, edgecolor="white", alpha=0.7, color="#2196F3")
axes[0, 0].axvline(latency.median(), color="red", linestyle="--", label=f"Median")
axes[0, 0].set_xlabel("Latency (μs)")
axes[0, 0].set_title("Latency Distribution")
axes[0, 0].legend()

# Log histogram
axes[0, 1].hist(np.log10(latency[latency > 0]), bins=100, edgecolor="white", alpha=0.7, color="#4CAF50")
axes[0, 1].set_xlabel("log₁₀(Latency μs)")
axes[0, 1].set_title("Log-Scale Distribution")

# CDF
sorted_lat = np.sort(latency)
cdf = np.arange(1, len(sorted_lat) + 1) / len(sorted_lat)
axes[1, 0].plot(sorted_lat, cdf, color="#2196F3", linewidth=2)
axes[1, 0].set_xlabel("Latency (μs)")
axes[1, 0].set_ylabel("Cumulative Probability")
axes[1, 0].set_title("Latency CDF")

# Tail analysis
survival = 1 - cdf
axes[1, 1].semilogy(sorted_lat, survival, color="#E53935", linewidth=2)
axes[1, 1].set_xlabel("Latency (μs)")
axes[1, 1].set_ylabel("P(Latency > x)")
axes[1, 1].set_title("Tail Distribution")

plt.tight_layout()
plt.show()
